# **Rantai Algoritma dan Pipeline (Algorithm Chains and Pipelines)**
Catatan ini menguraikan penerapan Pipeline dalam ekosistem machine learning untuk mengintegrasikan berbagai tahapan prapemrosesan data dan pelatihan model ke dalam satu alur kerja (workflow) yang kohesif. Fokus utama pembahasan mencakup penggabungan preprocessing dan cross-validation, mitigasi risiko data leakage (kebocoran data), serta optimasi parameter secara simultan menggunakan Grid Search.

## **Tujuan Pembelajaran**
Arsitektur Alur Kerja: Memahami rasionalitas di balik penggunaan alur kerja machine learning yang terdiri dari multi-tahapan.

Integrasi Pipeline: Mampu mengimplementasikan objek Pipeline untuk menyatukan tahapan prapemrosesan dan estimator akhir.

Mitigasi Data Leakage: Mengidentifikasi dan mencegah kebocoran informasi yang terjadi apabila prapemrosesan dieksekusi mendahului cross-validation.

Optimasi Lanjutan: Mengoperasikan GridSearchCV di dalam pipeline untuk melakukan tuning parameter secara bersamaan, hingga membandingkan performa antar-model yang berbeda.

# **Bagian 1: Persiapan Lingkungan Kerja (Environment Setup)**
Modul ini menggunakan pustaka standar komputasi dan memuat dataset bawaan (breast_cancer, diabetes, dan iris) untuk memastikan eksperimen dapat direproduksi tanpa ketergantungan pada berkas eksternal.

In [43]:
# Memuat pustaka komputasi dan visualisasi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Memuat dataset bawaan scikit-learn
from sklearn.datasets import load_breast_cancer, load_diabetes, load_iris

# Memuat modul evaluasi, validasi, dan metrik
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score

# Memuat modul prapemrosesan dan ekstraksi fitur
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PolynomialFeatures
from sklearn.feature_selection import SelectPercentile, f_regression
from sklearn.decomposition import PCA

# Memuat modul Pipeline
from sklearn.pipeline import Pipeline, make_pipeline

# Memuat algoritma machine learning
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier

# Konfigurasi parameter dasar
np.random.seed(42)
pd.set_option("display.max_columns", 100)

# **Bagian 2: Problematika Prapemrosesan Manual & Data Leakage**
Dalam skenario machine learning dunia nyata, data mentah jarang dapat langsung diumpankan ke dalam model. Data sering kali membutuhkan penskalaan (scaling), seleksi fitur, atau transformasi. Jika proses ini dilakukan secara terpisah sebelum cross-validation (CV), risiko terjadinya kebocoran data (data leakage) sangat tinggi.

Kebocoran data terjadi apabila scaler atau feature selector dilatih (menggunakan .fit()) pada seluruh himpunan data, termasuk lipatan validasi (validation fold). Akibatnya, model secara tidak langsung telah "mengintip" informasi dari data yang seharusnya belum pernah dilihat, sehingga menghasilkan skor evaluasi yang sangat optimis namun palsu.

In [44]:
# ==========================================
# CONTOH PENDEKATAN MANUAL (RENTAN LEAKAGE)
# ==========================================
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(cancer.data, cancer.target, random_state=0)

# Penskalaan dilakukan secara terpisah (Manual)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Definisi parameter untuk GridSearch
param_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "gamma": [0.001, 0.01, 0.1, 1, 10, 100]
}

# GridSearch dijalankan pada data yang SUDAH diskala seluruhnya (Pendekatan Naif)
grid_naive = GridSearchCV(SVC(), param_grid=param_grid, cv=5)
grid_naive.fit(X_train_scaled, y_train)

print(f"Skor CV Pendekatan Manual (Berpotensi Leakage): {grid_naive.best_score_:.2f}")
print(f"Skor Uji Pendekatan Manual: {grid_naive.score(X_test_scaled, y_test):.2f}")

Skor CV Pendekatan Manual (Berpotensi Leakage): 0.98
Skor Uji Pendekatan Manual: 0.97


# **Bagian 3: Implementasi Pipeline**
Pipeline menyelesaikan masalah data leakage dengan membungkus urutan transformasi data dan estimator akhir menjadi satu entitas (objek estimator tunggal). Saat cross-validation membelah data, Pipeline memastikan bahwa proses .fit() pada tahapan preprocessing hanya diaplikasikan pada lipatan latih (training fold), dan lipatan validasi hanya diubah menggunakan .transform().

In [45]:
# ==========================================
# PENDEKATAN MENGGUNAKAN PIPELINE (AMAN)
# ==========================================
pipe = Pipeline([
    ("scaler", MinMaxScaler()), # Tahap 1: Transformer
    ("svm", SVC())              # Tahap 2: Estimator Akhir
])

# Evaluasi standar menggunakan Pipeline
pipe.fit(X_train, y_train)
print(f"Skor Uji menggunakan Pipeline: {pipe.score(X_test, y_test):.2f}")

Skor Uji menggunakan Pipeline: 0.97


# **Bagian 4: Integrasi Pipeline dengan GridSearchCV**
Untuk mengoptimasi hiperparameter dari tahapan yang berada di dalam Pipeline, kita harus memberikan awalan berupa nama tahapan (step) diikuti dengan dua garis bawah (__) sebelum nama parameter model. Sintaksis ini memungkinkan GridSearchCV mengakses parameter spesifik di dalam antrean Pipeline.

In [46]:
# Sintaksis: nama_step__nama_parameter
param_grid_pipe = {
    "svm__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "svm__gamma": [0.001, 0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(pipe, param_grid=param_grid_pipe, cv=5)
grid.fit(X_train, y_train)

print("Parameter Terbaik:", grid.best_params_)
print(f"Skor CV Terbaik (Pipeline Terintegrasi): {grid.best_score_:.2f}")
print(f"Skor Uji Final: {grid.score(X_test, y_test):.2f}")

Parameter Terbaik: {'svm__C': 1, 'svm__gamma': 1}
Skor CV Terbaik (Pipeline Terintegrasi): 0.98
Skor Uji Final: 0.97


# **Bagian 5: Mengakses Atribut Eksekusi dan Penyetelan Simultan**
Setelah GridSearch menemukan model terbaik, kita sering kali perlu menginspeksi bobot model atau matriks transformasi. Atribut spesifik ini dapat diekstraksi melalui properti named_steps.

Selain itu, Pipeline memungkinkan kita untuk menjalankan iterasi tuning parameter pada tahap preprocessing dan model prediksi secara bersamaan (misalnya: mencari derajat polinomial paling ideal yang diselaraskan dengan parameter penalti regresi).

In [47]:
# 1. Mengakses Atribut Spesifik dari Pipeline Terbaik
pipe_logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
param_grid_logreg = {"logisticregression__C": [0.01, 0.1, 1, 10, 100]}

grid_logreg = GridSearchCV(pipe_logreg, param_grid_logreg, cv=5).fit(X_train, y_train)

# Ekstraksi komponen koefisien dari estimator model linier
logreg_step = grid_logreg.best_estimator_.named_steps["logisticregression"]
print(f"Dimensi Matriks Koefisien Regresi Logistik: {logreg_step.coef_.shape}")


# 2. Penyetelan Simultan: Prapemrosesan dan Model Regresi (Dataset Diabetes)
diabetes = load_diabetes()
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(diabetes.data, diabetes.target, random_state=0)

pipe_ridge = make_pipeline(StandardScaler(), PolynomialFeatures(), Ridge())

# Mencari derajat Polinomial optimal bersamaan dengan nilai Alpha untuk Ridge
param_grid_ridge = {
    "polynomialfeatures__degree": [1, 2, 3],
    "ridge__alpha": [0.001, 0.01, 0.1, 1, 10, 100]
}

grid_ridge = GridSearchCV(pipe_ridge, param_grid=param_grid_ridge, cv=5)
grid_ridge.fit(X_train_d, y_train_d)

print("\nKombinasi Parameter Preprocessing & Model Terbaik:", grid_ridge.best_params_)
print(f"Skor R-Squared Uji Final: {grid_ridge.score(X_test_d, y_test_d):.2f}")

Dimensi Matriks Koefisien Regresi Logistik: (1, 30)

Kombinasi Parameter Preprocessing & Model Terbaik: {'polynomialfeatures__degree': 1, 'ridge__alpha': 10}
Skor R-Squared Uji Final: 0.36


# **Bagian 6: Komparasi Model Skala Besar Menggunakan GridSearch**
Pipeline yang digabungkan dengan antrean daftar dictionary (param_grid berbasis himpunan) mampu mengevaluasi algoritma machine learning yang sama sekali berbeda dalam satu proses eksekusi tunggal. Pendekatan ini sangat efektif untuk mengidentifikasi kombinasi arsitektur mana (beserta persyaratan prapemrosesannya) yang memberikan performa absolut tertinggi.

In [48]:
# Membandingkan arsitektur Support Vector Machine vs Random Forest
iris = load_iris()
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(iris.data, iris.target, random_state=0)

# Pipeline dasar sebagai cetak biru awal
pipe_model_selection = Pipeline([
    ("preprocessing", StandardScaler()),
    ("classifier", SVC())
])

# Definisi ruang pencarian untuk dua algoritma berbeda
param_grid_model_selection = [
    {   # Ruang pencarian untuk SVM (Membutuhkan Penskalaan)
        "classifier": [SVC()],
        "preprocessing": [StandardScaler(), None],
        "classifier__C": [0.1, 1, 10],
        "classifier__gamma": [0.001, 0.01, 0.1, 1]
    },
    {   # Ruang pencarian untuk Random Forest (Tidak membutuhkan Penskalaan)
        "classifier": [RandomForestClassifier(n_estimators=100, random_state=0)],
        "preprocessing": [None], # Melewati tahap scaling
        "classifier__max_features": [1, 2, 3]
    }
]

grid_model_selection = GridSearchCV(pipe_model_selection, param_grid_model_selection, cv=5)
grid_model_selection.fit(X_train_i, y_train_i)

print("Kandidat Algoritma Terbaik:")
print(grid_model_selection.best_params_)
print(f"Skor Final Evaluasi Uji: {grid_model_selection.score(X_test_i, y_test_i):.2f}")

Kandidat Algoritma Terbaik:
{'classifier': SVC(), 'classifier__C': 10, 'classifier__gamma': 0.1, 'preprocessing': None}
Skor Final Evaluasi Uji: 0.97
